In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Вимірювання кубітів

<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблено з використанням наведених нижче залежностей.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Щоб отримати інформацію про стан кубіта, можна _виміряти_ його на [класичний біт](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit). У Qiskit вимірювання виконуються в обчислювальному базисі, тобто в однокубітному базисі Паулі-$Z$. Тому вимірювання дає результат 0 або 1, залежно від перекриття з власними станами Паулі-$Z$ $|0\rangle$ і $|1\rangle$:

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## Вимірювання всередині схеми
Вимірювання всередині схеми є ключовим компонентом динамічних схем. До версії `qiskit-ibm-runtime` v0.43.0 інструкція `measure` була єдиним способом вимірювання в Qiskit. Проте вимірювання всередині схеми мають інші вимоги до налаштування, ніж _термінальні_ вимірювання (ті, що виконуються в кінці схеми). Наприклад, при налаштуванні вимірювання всередині схеми потрібно враховувати тривалість інструкції, оскільки довші інструкції призводять до більш шумних схем. Для термінальних вимірювань тривалість інструкції не має значення, бо після них більше немає жодних інструкцій.

У версії `qiskit-ibm-runtime` v0.43.0 було введено інструкцію `MidCircuitMeasure`. Як випливає з назви, це нова інструкція вимірювання, оптимізована для вимірювань усередині схеми на QPU IBM&reg;.

> **Note:** Інструкція `MidCircuitMeasure` відповідає інструкції `measure_2`, яка вказана у `supported_instructions` backend-у. Однак `measure_2` підтримується не всіма backend-ами. Використовуй `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)`, щоб знайти backend-и, які її підтримують. У майбутньому можуть бути додані нові вимірювання, але це не гарантується.

## Додавання вимірювання до схеми
Є кілька способів додати вимірювання до схеми:

### Метод `QuantumCircuit.measure`
Використовуй метод [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure), щоб виміряти [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class).

Приклади:

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### Клас `Measure`
Клас Qiskit [Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure) вимірює вказані кубіти.

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

<span id="midcircuit"></span>
### `MidCircuitMeasure` method


Use `MidCircuitMeasure` to apply a mid-circuit measurement (requires `qiskit-ibm-runtime` v0.43.0 or later).  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [6]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


### Метод `QuantumCircuit.measure_all`
Щоб виміряти всі кубіти у відповідні класичні біти, використовуй метод [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all). За замовчуванням цей метод додає нові класичні біти в `ClassicalRegister` для збереження результатів вимірювань.